# 역설계 실습

**Inverse Design**

조성에서 물성을 예측하는 방향과 반대로, 원하는 물성에서 후보 구조나 조성을 찾는 접근.

소재 분야에서 이해하기: 높은 유전율이라는 목표를 만족할 후보를 생성한다.

이 노트북은 개념을 직접 돌려보기 위한 예제입니다. 데이터는 실제 측정값이 아니라 개념 확인용으로
생성한 값이므로, 결과 수치를 연구 결론으로 쓰지 마세요. 위에서부터 순서대로 실행하세요.
그림의 축 이름은 기본 폰트에 한글 글리프가 없어 영문으로 적었습니다.

참고 자료: [소재 발견용 그래프 신경망 연구](https://www.nature.com/articles/s41586-023-06735-9)

## 1. 목표 물성에서 조건을 거꾸로 찾기

순방향 대리 모델을 학습한 뒤, 목표값을 만족하는 입력을 최적화로 찾습니다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)
plt.rcParams['figure.figsize'] = (7, 4)

def forward(x):
    """조성 2개 -> (밴드갭 eV, 형성에너지 eV/atom) 가상 관계"""
    a, b = x[..., 0], x[..., 1]
    band_gap = 0.6 + 2.4 * a - 1.1 * b ** 2 + 0.8 * a * b
    formation = -1.8 + 0.9 * a ** 2 + 0.4 * b - 0.6 * a * b
    return np.stack([band_gap, formation], axis=-1)

train_x = rng.random((400, 2))
train_y = forward(train_x) + rng.normal(0, 0.02, (400, 2))
print('학습 데이터 %d건, 밴드갭 범위 %.2f ~ %.2f eV' % (len(train_x), train_y[:, 0].min(), train_y[:, 0].max()))

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from scipy.optimize import differential_evolution

model = RandomForestRegressor(n_estimators=300, random_state=0).fit(train_x, train_y)
target_gap, max_formation = 1.5, -1.5

def loss(x):
    prediction = model.predict(np.array(x)[None, :])[0]
    penalty = max(0.0, prediction[1] - max_formation) * 10      # 형성 에너지 제약
    return (prediction[0] - target_gap) ** 2 + penalty

result = differential_evolution(loss, [(0, 1), (0, 1)], seed=0, tol=1e-8)
found = result.x
print('찾은 조성 %s' % np.round(found, 3))
print('대리 모델 예측 밴드갭 %.3f eV, 형성에너지 %.3f eV/atom' % tuple(model.predict(found[None, :])[0]))
print('참 관계로 검증      밴드갭 %.3f eV, 형성에너지 %.3f eV/atom' % tuple(forward(found)))

In [ ]:
gx, gy = np.meshgrid(np.linspace(0, 1, 200), np.linspace(0, 1, 200))
points = np.stack([gx, gy], axis=-1)
truth = forward(points)
plt.contour(gx, gy, truth[..., 0], levels=[target_gap], colors='blue')
plt.contourf(gx, gy, truth[..., 1] <= max_formation, levels=[0.5, 1.5], colors=['lightgreen'], alpha=0.5)
plt.scatter(*found, c='red', s=60, zorder=5, label='inverse design result')
plt.xlabel('component a'); plt.ylabel('component b')
plt.title('blue line: target band gap, green: allowed formation energy')
plt.legend(); plt.show()
print('목표를 만족하는 해가 하나가 아닐 수 있습니다(파란 선 전체). 어떤 해를 고를지는 추가 기준이 필요합니다.')
print('또 대리 모델의 오차 때문에 참 관계로는 목표를 살짝 벗어날 수 있어, 검증이 반드시 필요합니다.')

---

셀의 숫자를 바꿔가며 다시 실행해보면 개념이 더 분명해집니다. 용어 사전으로 돌아가려면
[소재·AI 용어 사전](https://forum.rnddata.org/glossary/#inverse-design)을 여세요.